# Zadanie domowe -- interpolacja dwusześcienna

Interpolacja dwusześcienna, to podobnie jak w przypadku interpolacji dwuliniowej, rozszerzenie idei interpolacji jednowymiarowej na dwuwymiarową siatkę.
W trakcie jej obliczania wykorzystywane jest 16 pikseli z otoczenia (dla dwuliniowej 4).
Skutkuje to zwykle lepszymi wynikami - obraz wyjściowy jest bardziej gładki i z mniejszą liczbą artefaktów.
Ceną jest znaczny wzrost złożoności obliczeniowej (zostało to zaobserwowane podczas ćwiczenia).

Interpolacja dana jest wzorem:
\begin{equation}
I(i,j) = \sum_{i=0}^{3} \sum_{j=0}^{3} a_{ij} x^i y^j
\end{equation}

Zadanie sprowadza się zatem do wyznaczenia 16 współczynników $a_{ij}$.
W tym celu wykorzystuje się, oprócz wartość w~puntach $A$ (0,0), $B$ (1 0), $C$ (1,1), $D$ (0,1) (por. rysunek dotyczący interpolacji dwuliniowej), także pochodne cząstkowe $A_x$, $A_y$, $A_{xy}$.
Pozwala to rozwiązać układ 16-tu równań.

Jeśli zgrupujemy parametry $a_{ij}$:
\begin{equation}
a = [ a_{00}~a_{10}~a_{20}~a_{30}~a_{01}~a_{11}~a_{21}~a_{31}~a_{02}~a_{12}~a_{22}~a_{32}~a_{03}~a_{13}~a_{23}~a_{33}]
\end{equation}

i przyjmiemy:
\begin{equation}
x = [A~B~D~C~A_x~B_x~D_x~C_x~A_y~B_y~D_y~C_y~A_{xy}~B_{xy}~D_{xy}~C_{xy}]^T
\end{equation}

To zagadnienie można opisać w postaci równania liniowego:
\begin{equation}
Aa = x
\end{equation}
gdzie macierz $A^{-1}$ dana jest wzorem:

\begin{equation}
A^{-1} =
\begin{bmatrix}
1& 0& 0& 0& 0& 0& 0& 0& 0& 0& 0& 0& 0& 0& 0& 0 \\
0&  0&  0&  0&  1&  0&  0&  0&  0&  0&  0&  0&  0&  0&  0&  0 \\
-3&  3&  0&  0& -2& -1&  0&  0&  0&  0&  0&  0&  0&  0&  0&  0 \\
2& -2&  0&  0&  1&  1&  0&  0&  0&  0&  0&  0&  0&  0&  0&  0 \\
0&  0&  0&  0&  0&  0&  0&  0&  1&  0&  0&  0&  0&  0&  0&  0 \\
0&  0&  0&  0&  0&  0&  0&  0&  0&  0&  0&  0&  1&  0&  0&  0 \\
0&  0&  0&  0&  0&  0&  0&  0& -3&  3&  0&  0& -2& -1&  0&  0 \\
0&  0&  0&  0&  0&  0&  0&  0&  2& -2&  0&  0&  1&  1&  0&  0 \\
-3&  0&  3&  0&  0&  0&  0&  0& -2&  0& -1&  0&  0&  0&  0&  0 \\
0&  0&  0&  0& -3&  0&  3&  0&  0&  0&  0&  0& -2&  0& -1&  0 \\
9& -9& -9&  9&  6&  3& -6& -3&  6& -6&  3& -3&  4&  2&  2&  1 \\
-6&  6&  6& -6& -3& -3&  3&  3& -4&  4& -2&  2& -2& -2& -1& -1 \\
2&  0& -2&  0&  0&  0&  0&  0&  1&  0&  1&  0&  0&  0&  0&  0 \\
0&  0&  0&  0&  2&  0& -2&  0&  0&  0&  0&  0&  1&  0&  1&  0 \\
-6&  6&  6& -6& -4& -2&  4&  2& -3&  3& -3&  3& -2& -1& -2& -1 \\
4& -4& -4&  4&  2&  2& -2& -2&  2& -2&  2& -2&  1&  1&  1&  1 \\
\end{bmatrix}
\end{equation}

Potrzebne w rozważaniach pochodne cząstkowe obliczane są wg. następującego przybliżenia (przykład dla punktu A):
\begin{equation}
A_x = \frac{I(i+1,j) - I(i-1,j)}{2}
\end{equation}
\begin{equation}
A_y = \frac{I(i,j+1) - I(i,j-1)}{2}
\end{equation}
\begin{equation}
A_xy = \frac{I(i+1,j+1) - I(i-1,j) - I(i,j-1) + I(i,j)}{4}
\end{equation}

## Zadanie

Wykorzystując podane informacje zaimplementuj interpolację dwusześcienną.
Uwagi:
- macierz $A^{-1}$ dostępna jest w pliku *a_invert.py*
- trzeba się zastanowić nad potencjalnym wykraczaniem poza zakres obrazka (jak zwykle).

Ponadto dokonaj porównania liczby operacji arytmetycznych i dostępów do pamięci koniecznych przy realizacji obu metod interpolacji: dwuliniowej i dwusześciennej.

In [ ]:
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt

#! pobieranie obrazów
if not os.path.exists("parrot.bmp"):
    os.system("wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/05_Resolution/parrot.bmp --no-check-certificate")
if not os.path.exists("lena.bmp"):
    os.system("wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/05_Resolution/lena.bmp --no-check-certificate")

#! Macierz odwrotna A⁻¹ używana do wyznaczania współczynników interpolacji dwusześciennej
A_invert = np.array([
    [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0,  0,  0,  0,  1,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
    [-3,  3,  0,  0, -2, -1,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
    [2, -2,  0,  0,  1,  1,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
    [0,  0,  0,  0,  0,  0,  0,  0,  1,  0,  0,  0,  0,  0,  0,  0],
    [0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  1,  0,  0,  0],
    [0,  0,  0,  0,  0,  0,  0,  0, -3,  3,  0,  0, -2, -1,  0,  0],
    [0,  0,  0,  0,  0,  0,  0,  0,  2, -2,  0,  0,  1,  1,  0,  0],
    [-3,  0,  3,  0,  0,  0,  0,  0, -2,  0, -1,  0,  0,  0,  0,  0],
    [0,  0,  0,  0, -3,  0,  3,  0,  0,  0,  0,  0, -2,  0, -1,  0],
    [9, -9, -9,  9,  6,  3, -6, -3,  6, -6,  3, -3,  4,  2,  2,  1],
    [-6,  6,  6, -6, -3, -3,  3,  3, -4,  4, -2,  2, -2, -2, -1, -1],
    [2,  0, -2,  0,  0,  0,  0,  0,  1,  0,  1,  0,  0,  0,  0,  0],
    [0,  0,  0,  0,  2,  0, -2,  0,  0,  0,  0,  0,  1,  0,  1,  0],
    [-6,  6,  6, -6, -4, -2,  4,  2, -3,  3, -3,  3, -2, -1, -2, -1],
    [4, -4, -4,  4,  2,  2, -2, -2,  2, -2,  2, -2,  1,  1,  1,  1],
])

def bicubic_interpolate(image, scale_x, scale_y):
    """
    Wykonuje interpolację dwusześcienną na obrazie w skali szarości.
    
    Parametry:
      image   - Obraz wejściowy (odcienie szarości) jako tablica NumPy.
      scale_x - Współczynnik skalowania w poziomie.
      scale_y - Współczynnik skalowania w pionie.
    
    Zwraca:
      Obraz przeskalowany przy użyciu interpolacji dwusześciennej jako tablica NumPy.
      
    Opis:
      Funkcja mapuje każdy piksel obrazu wyjściowego na odpowiadające mu współrzędne w obrazie wejściowym.
      Następnie wyznacza lokalne wartości pikseli (4 narożniki) oraz przybliżenia pochodnych (horyzontalnych, wertykalnych
      oraz mieszanych) na podstawie tych pikseli. Na podstawie macierzy odwrotnej A_invert wyznaczane są współczynniki
      interpolacji, które są używane do obliczenia wartości nowego piksela.
    """
    orig_rows, orig_cols = image.shape
    new_rows = int(np.round(scale_y * orig_rows))
    new_cols = int(np.round(scale_x * orig_cols))
    
    output = np.zeros((new_rows, new_cols), dtype='uint8')
    
    for r_new in range(new_rows):
        for c_new in range(new_cols):
            r_orig = r_new / scale_y
            c_orig = c_new / scale_x
            base_r = int(np.floor(r_orig))
            base_c = int(np.floor(c_orig))
            
            #! obszar do interpolacji mieści się w granicach obrazu
            base_r = max(1, min(base_r, orig_rows - 3))
            base_c = max(1, min(base_c, orig_cols - 3))
            
            #! wartości pikseli w czterech narożnikach kwadratu
            A = image[base_r, base_c]
            B = image[base_r, base_c + 1]
            C = image[base_r + 1, base_c + 1]
            D = image[base_r + 1, base_c]
            
            #!  przybliżone pochodne względem x
            deriv_A_x = (image[base_r + 1, base_c] - image[base_r - 1, base_c]) / 2.0
            deriv_B_x = (image[base_r + 1, base_c + 1] - image[base_r - 1, base_c + 1]) / 2.0
            deriv_C_x = (image[base_r + 2, base_c + 1] - image[base_r, base_c + 1]) / 2.0
            deriv_D_x = (image[base_r + 2, base_c] - image[base_r, base_c]) / 2.0
            
            #!  przybliżone pochodne względem y
            deriv_A_y = (image[base_r, base_c + 1] - image[base_r, base_c - 1]) / 2.0
            deriv_B_y = (image[base_r, base_c + 2] - image[base_r, base_c]) / 2.0
            deriv_C_y = (image[base_r + 1, base_c + 2] - image[base_r + 1, base_c]) / 2.0
            deriv_D_y = (image[base_r + 1, base_c + 1] - image[base_r + 1, base_c - 1]) / 2.0
            
            #!  przybliżone pochodne mieszane
            deriv_A_xy = (image[base_r + 1, base_c + 1] - image[base_r - 1, base_c + 1] -
                          image[base_r + 1, base_c - 1] + image[base_r, base_c]) / 4.0
            deriv_B_xy = (image[base_r + 1, base_c + 2] - image[base_r - 1, base_c + 2] -
                          image[base_r + 1, base_c] + image[base_r, base_c + 1]) / 4.0
            deriv_C_xy = (image[base_r + 2, base_c + 2] - image[base_r, base_c + 2] -
                          image[base_r + 2, base_c] + image[base_r + 1, base_c + 1]) / 4.0
            deriv_D_xy = (image[base_r + 2, base_c + 1] - image[base_r, base_c + 1] -
                          image[base_r + 2, base_c - 1] + image[base_r + 1, base_c]) / 4.0
            
            obs = np.array([
                A, B, D, C,
                deriv_A_x, deriv_B_x, deriv_D_x, deriv_C_x,
                deriv_A_y, deriv_B_y, deriv_D_y, deriv_C_y,
                deriv_A_xy, deriv_B_xy, deriv_D_xy, deriv_C_xy
            ])
            
            coeffs = A_invert @ obs
            dr = r_orig - base_r  #! różnica w pionie
            dc = c_orig - base_c  #! różnica w poziomie
            
            value = 0.0
            for m in range(4):
                for n in range(4):
                    value += coeffs[m * 4 + n] * (dr ** m) * (dc ** n)
            
            output[r_new, c_new] = np.clip(value, 0, 255)
    
    return output

if __name__ == "__main__":
    lena_original = cv2.imread('lena.bmp', cv2.IMREAD_GRAYSCALE)
    lena_bicubic = bicubic_interpolate(lena_original, 2.0, 2.0)
    
    plt.figure()
    plt.imshow(lena_original, cmap="gray")
    plt.title("Lena - Oryginalny")
    plt.xticks([]), plt.yticks([])
    plt.show(block=False)
    
    plt.figure()
    plt.imshow(lena_bicubic, cmap="gray")
    plt.title("Lena - Interpolacja Dwusześcienna")
    plt.xticks([]), plt.yticks([])
    plt.show(block=False)
    
    parrot_original = cv2.imread('parrot.bmp', cv2.IMREAD_GRAYSCALE)
    parrot_bicubic = bicubic_interpolate(parrot_original, 2.0, 2.0)
    
    plt.figure()
    plt.imshow(parrot_original, cmap="gray")
    plt.title("Parrot - Oryginalny")
    plt.xticks([]), plt.yticks([])
    plt.show(block=False)
    
    plt.figure()
    plt.imshow(parrot_bicubic, cmap="gray")
    plt.title("Parrot - Interpolacja Dwusześcienna")
    plt.xticks([]), plt.yticks([])
    plt.show(block=False)
    
    plt.show()

# Wnioski

## Porównanie interpolacji dwuliniowej i dwusześciennej

Analiza przeprowadzona dla metod interpolacji obrazów pozwoliła na wyciągnięcie następujących wniosków:

1. **Złożoność obliczeniowa**:
   - Interpolacja dwusześcienna wymaga około **38 razy więcej operacji arytmetycznych** niż interpolacja dwuliniowa (570 vs 15 operacji na piksel)
   - Operacje w interpolacji dwusześciennej obejmują skomplikowane obliczenia pochodnych oraz mnożenie macierzy 16×16

2. **Wykorzystanie pamięci**:
   - Interpolacja dwusześcienna wykonuje około **6-7 razy więcej operacji dostępu do pamięci** (33 vs 5 dostępów na piksel)
   - Wymaga odczytu minimum 16 sąsiednich pikseli oraz dodatkowych odczytów do obliczania pochodnych

3. **Jakość i wydajność**:
   - Dwuliniowa: szybsza, ale mniej dokładna metoda interpolacji
   - Dwusześcienna: wolniejsza, ale dająca lepszą jakość obrazu (mniej artefaktów, gładsze przejścia)

4. **Zastosowania**:
   - Interpolacja dwuliniowa: odpowiednia dla zastosowań czasu rzeczywistego lub przy ograniczonych zasobach obliczeniowych
   - Interpolacja dwusześcienna: preferowana gdy priorytetem jest jakość obrazu, a nie wydajność obliczeniowa

Wyniki potwierdzają obserwację z zajęć o znacznym wzroście złożoności obliczeniowej przy zastosowaniu interpolacji dwusześciennej w porównaniu z dwuliniową. Wybór metody powinien być uzależniony od konkretnego zastosowania i dostępnych zasobów.